# 03 — Generate Initial Subscription Snapshot

## Purpose

This notebook generates a deterministic subscription snapshot for a subscription-based business.

The dataset defines contractual pricing, discounts, billing schedules, usage allowances, overage rates, subscription lifecycle statuses, and payment terms. These attributes provide the expected-revenue baseline required for downstream revenue-leakage detection.

## Dataset Overview

- 6,000 subscriptions
- 5,000 referenced customers
- 1,000 customers with multiple subscriptions
- Four service plans
- Monthly and annual billing frequencies
- Active, Paused, and Cancelled lifecycle statuses
- Temporary contract discounts
- Usage allowances and overage pricing

## Data Quality Controls

- Explicit source and target schemas
- Unique subscription business keys
- Customer referential integrity
- Customer-signup temporal integrity
- Pricing and discount reconciliation
- Billing-amount reconciliation
- Date-range validation
- Domain-value validation
- Multi-subscription relationship validation

## Input

`/Volumes/workspace/revenue_leakage_bronze/landing/crm/customers/initial_load`

## Target

`/Volumes/workspace/revenue_leakage_bronze/landing/subscription_system/subscriptions/initial_load`

## 1. Configuration and Schemas

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    DateType,
    TimestampType,
    IntegerType,
    DecimalType,
    BooleanType,
)


INITIAL_CUSTOMER_COUNT = 5000
EXPECTED_SUBSCRIPTION_COUNT = 6000
EXPECTED_MULTI_SUBSCRIPTION_CUSTOMERS = 1000

SNAPSHOT_DATE = "2026-08-15"

LANDING_PATH = (
    "/Volumes/workspace/"
    "revenue_leakage_bronze/"
    "landing"
)

CUSTOMERS_INITIAL_PATH = (
    f"{LANDING_PATH}/crm/customers/initial_load"
)

SUBSCRIPTIONS_PATH = (
    f"{LANDING_PATH}/subscription_system/subscriptions"
)

SUBSCRIPTIONS_INITIAL_PATH = (
    f"{SUBSCRIPTIONS_PATH}/initial_load"
)

CUSTOMER_REFERENCE_SCHEMA = StructType([
    StructField("customer_id", StringType(), False),
    StructField("signup_date", DateType(), True),
])

SUBSCRIPTION_SCHEMA = StructType([
    StructField("subscription_id", StringType(), False),
    StructField("customer_id", StringType(), False),
    StructField("plan_id", StringType(), False),
    StructField("plan_name", StringType(), False),
    StructField("start_date", DateType(), False),
    StructField("end_date", DateType(), True),
    StructField("subscription_status", StringType(), False),
    StructField("billing_frequency", StringType(), False),
    StructField("billing_day", IntegerType(), False),
    StructField(
        "base_monthly_price",
        DecimalType(10, 2),
        False
    ),
    StructField(
        "discount_percentage",
        DecimalType(5, 2),
        False
    ),
    StructField(
        "contracted_monthly_price",
        DecimalType(10, 2),
        False
    ),
    StructField(
        "contracted_billing_amount",
        DecimalType(12, 2),
        False
    ),
    StructField("discount_start_date", DateType(), True),
    StructField("discount_end_date", DateType(), True),
    StructField("included_usage_units", IntegerType(), False),
    StructField(
        "overage_unit_price",
        DecimalType(10, 4),
        False
    ),
    StructField("payment_terms_days", IntegerType(), False),
    StructField("auto_renew", BooleanType(), False),
    StructField("currency", StringType(), False),
    StructField("operation", StringType(), False),
    StructField("event_timestamp", TimestampType(), False),
    StructField("snapshot_date", DateType(), False),
])

## 2. Load the Customer Reference Dataset

Load the customer business keys and signup dates required for referential-integrity and temporal-integrity validation.

In [0]:
customer_reference_df = (
    spark.read
    .schema(CUSTOMER_REFERENCE_SCHEMA)
    .json(CUSTOMERS_INITIAL_PATH)
    .select(
        "customer_id",
        F.col("signup_date").alias("customer_signup_date")
    )
    .distinct()
)

customer_key_count = customer_reference_df.count()

invalid_customer_signup_count = (
    customer_reference_df
    .filter(F.col("customer_signup_date").isNull())
    .count()
)

assert customer_key_count == INITIAL_CUSTOMER_COUNT
assert invalid_customer_signup_count == 0

print(f"Available customer keys: {customer_key_count:,}")

print(
    "Invalid customer signup dates: "
    f"{invalid_customer_signup_count:,}"
)

## 3. Generate the Initial Subscription Snapshot

Generate subscription records linked to valid customers. Assign deterministic service plans, pricing terms, discounts, billing schedules, usage allowances, payment terms, and lifecycle attributes.

In [0]:
subscriptions_initial_df = (
    spark.range(1, EXPECTED_SUBSCRIPTION_COUNT + 1)
    .withColumnRenamed("id", "subscription_number")

    .withColumn(
        "subscription_id",
        F.format_string(
            "S%07d",
            F.col("subscription_number")
        )
    )

    .withColumn(
        "customer_number",
        F.pmod(
            F.col("subscription_number") - 1,
            F.lit(INITIAL_CUSTOMER_COUNT)
        ) + 1
    )

    .withColumn(
        "customer_id",
        F.format_string(
            "C%06d",
            F.col("customer_number")
        )
    )

    .join(
        customer_reference_df,
        on="customer_id",
        how="inner"
    )

    .withColumn(
        "plan_bucket",
        F.pmod(
            F.col("subscription_number") * 7 + 11,
            F.lit(100)
        )
    )

    .withColumn(
        "plan_id",
        F.when(
            F.col("plan_bucket") < 50,
            "PLAN_STARTER"
        )
        .when(
            F.col("plan_bucket") < 80,
            "PLAN_GROWTH"
        )
        .when(
            F.col("plan_bucket") < 95,
            "PLAN_PROFESSIONAL"
        )
        .otherwise("PLAN_ENTERPRISE")
    )

    .withColumn(
        "plan_name",
        F.when(
            F.col("plan_id") == "PLAN_STARTER",
            "Starter"
        )
        .when(
            F.col("plan_id") == "PLAN_GROWTH",
            "Growth"
        )
        .when(
            F.col("plan_id") == "PLAN_PROFESSIONAL",
            "Professional"
        )
        .otherwise("Enterprise")
    )

    .withColumn(
        "base_monthly_price",
        F.when(
            F.col("plan_id") == "PLAN_STARTER",
            29.00
        )
        .when(
            F.col("plan_id") == "PLAN_GROWTH",
            79.00
        )
        .when(
            F.col("plan_id") == "PLAN_PROFESSIONAL",
            149.00
        )
        .otherwise(399.00)
        .cast(DecimalType(10, 2))
    )

    .withColumn(
        "included_usage_units",
        F.when(
            F.col("plan_id") == "PLAN_STARTER",
            1000
        )
        .when(
            F.col("plan_id") == "PLAN_GROWTH",
            5000
        )
        .when(
            F.col("plan_id") == "PLAN_PROFESSIONAL",
            15000
        )
        .otherwise(50000)
        .cast("int")
    )

    .withColumn(
        "overage_unit_price",
        F.when(
            F.col("plan_id") == "PLAN_STARTER",
            0.0500
        )
        .when(
            F.col("plan_id") == "PLAN_GROWTH",
            0.0400
        )
        .when(
            F.col("plan_id") == "PLAN_PROFESSIONAL",
            0.0300
        )
        .otherwise(0.0200)
        .cast(DecimalType(10, 4))
    )

    .withColumn(
        "discount_percentage",
        F.when(
            F.col("subscription_number") % 17 == 0,
            15.00
        )
        .when(
            F.col("subscription_number") % 11 == 0,
            10.00
        )
        .when(
            F.col("subscription_number") % 7 == 0,
            5.00
        )
        .otherwise(0.00)
        .cast(DecimalType(5, 2))
    )

    .withColumn(
        "contracted_monthly_price",
        F.round(
            F.col("base_monthly_price")
            * (
                1
                - F.col("discount_percentage") / 100
            ),
            2
        ).cast(DecimalType(10, 2))
    )

    .withColumn(
        "billing_frequency",
        F.when(
            F.col("subscription_number") % 5 == 0,
            "Annual"
        ).otherwise("Monthly")
    )

    .withColumn(
        "contracted_billing_amount",
        F.when(
            F.col("billing_frequency") == "Annual",
            F.col("contracted_monthly_price") * 12
        )
        .otherwise(
            F.col("contracted_monthly_price")
        )
        .cast(DecimalType(12, 2))
    )

    .withColumn(
        "billing_day",
        (
            F.pmod(
                F.col("subscription_number") - 1,
                F.lit(28)
            ) + 1
        ).cast("int")
    )

    .withColumn(
        "start_date",
        F.least(
            F.date_add(
                F.col("customer_signup_date"),
                F.pmod(
                    F.col("subscription_number") * 19,
                    F.lit(366)
                ).cast("int")
            ),
            F.lit(SNAPSHOT_DATE).cast("date")
        )
    )

    .withColumn(
        "snapshot_date",
        F.lit(SNAPSHOT_DATE).cast("date")
    )

    .withColumn(
        "subscription_status",
        F.when(
            F.col("subscription_number") % 23 == 0,
            "Cancelled"
        )
        .when(
            F.col("subscription_number") % 29 == 0,
            "Paused"
        )
        .otherwise("Active")
    )

    .withColumn(
        "end_date",
        F.when(
            F.col("subscription_status") == "Cancelled",
            F.least(
                F.date_add(
                    F.col("start_date"),
                    (
                        90
                        + F.pmod(
                            F.col("subscription_number") * 13,
                            F.lit(360)
                        )
                    ).cast("int")
                ),
                F.col("snapshot_date")
            )
        ).otherwise(F.lit(None).cast("date"))
    )

    .withColumn(
        "discount_start_date",
        F.when(
            F.col("discount_percentage") > 0,
            F.col("start_date")
        ).otherwise(F.lit(None).cast("date"))
    )

    .withColumn(
        "discount_end_date",
        F.when(
            F.col("discount_percentage") > 0,
            F.date_add(F.col("start_date"), 180)
        ).otherwise(F.lit(None).cast("date"))
    )

    .withColumn(
        "payment_terms_days",
        F.when(
            F.col("subscription_number") % 10 == 0,
            45
        )
        .when(
            F.col("subscription_number") % 3 == 0,
            15
        )
        .otherwise(30)
        .cast("int")
    )

    .withColumn(
        "auto_renew",
        F.when(
            F.col("subscription_status") == "Cancelled",
            F.lit(False)
        ).otherwise(
            F.col("subscription_number") % 9 != 0
        )
    )

    .withColumn("currency", F.lit("USD"))
    .withColumn("operation", F.lit("INSERT"))

    .withColumn(
        "event_timestamp",
        F.col("start_date").cast("timestamp")
    )

    .select(
        "subscription_id",
        "customer_id",
        "plan_id",
        "plan_name",
        "start_date",
        "end_date",
        "subscription_status",
        "billing_frequency",
        "billing_day",
        "base_monthly_price",
        "discount_percentage",
        "contracted_monthly_price",
        "contracted_billing_amount",
        "discount_start_date",
        "discount_end_date",
        "included_usage_units",
        "overage_unit_price",
        "payment_terms_days",
        "auto_renew",
        "currency",
        "operation",
        "event_timestamp",
        "snapshot_date"
    )
)

## 4. Validate the Generated Subscription Dataset

Validate subscription uniqueness, customer relationships, commercial terms, lifecycle dates, allowed domain values, and multi-subscription behavior before persistence.

In [0]:
actual_subscription_count = (
    subscriptions_initial_df.count()
)

distinct_subscription_count = (
    subscriptions_initial_df
    .select("subscription_id")
    .distinct()
    .count()
)

customer_coverage_count = (
    subscriptions_initial_df
    .select("customer_id")
    .distinct()
    .count()
)

orphan_customer_count = (
    subscriptions_initial_df
    .select("customer_id")
    .distinct()
    .join(
        customer_reference_df.select("customer_id"),
        on="customer_id",
        how="left_anti"
    )
    .count()
)

subscription_before_customer_signup_count = (
    subscriptions_initial_df
    .join(
        customer_reference_df,
        on="customer_id",
        how="inner"
    )
    .filter(
        F.col("start_date")
        < F.col("customer_signup_date")
    )
    .count()
)

null_business_key_count = (
    subscriptions_initial_df
    .filter(
        F.col("subscription_id").isNull()
        | F.col("customer_id").isNull()
        | F.col("plan_id").isNull()
    )
    .count()
)

invalid_price_count = (
    subscriptions_initial_df
    .filter(
        (F.col("base_monthly_price") <= 0)
        | (F.col("contracted_monthly_price") <= 0)
        | (F.col("contracted_billing_amount") <= 0)
        | (F.col("discount_percentage") < 0)
        | (F.col("discount_percentage") > 100)
        | (F.col("overage_unit_price") < 0)
        | (F.col("included_usage_units") <= 0)
    )
    .count()
)

invalid_domain_value_count = (
    subscriptions_initial_df
    .filter(
        ~F.col("plan_id").isin(
            "PLAN_STARTER",
            "PLAN_GROWTH",
            "PLAN_PROFESSIONAL",
            "PLAN_ENTERPRISE"
        )
        | ~F.col("subscription_status").isin(
            "Active",
            "Paused",
            "Cancelled"
        )
        | ~F.col("billing_frequency").isin(
            "Monthly",
            "Annual"
        )
        | ~F.col("payment_terms_days").isin(
            15,
            30,
            45
        )
        | (F.col("billing_day") < 1)
        | (F.col("billing_day") > 28)
        | (F.col("currency") != "USD")
        | (F.col("operation") != "INSERT")
    )
    .count()
)

pricing_mismatch_count = (
    subscriptions_initial_df
    .filter(
        F.abs(
            F.col("contracted_monthly_price")
            - F.round(
                F.col("base_monthly_price")
                * (
                    1
                    - F.col("discount_percentage") / 100
                ),
                2
            )
        ) > 0.01
    )
    .count()
)

billing_amount_mismatch_count = (
    subscriptions_initial_df
    .filter(
        F.abs(
            F.col("contracted_billing_amount")
            - F.when(
                F.col("billing_frequency") == "Annual",
                F.col("contracted_monthly_price") * 12
            ).otherwise(
                F.col("contracted_monthly_price")
            )
        ) > 0.01
    )
    .count()
)

invalid_date_count = (
    subscriptions_initial_df
    .filter(
        (F.col("start_date") > F.col("snapshot_date"))

        | (
            F.col("end_date").isNotNull()
            & (
                F.col("end_date")
                < F.col("start_date")
            )
        )

        | (
            (
                F.col("subscription_status")
                == "Cancelled"
            )
            & F.col("end_date").isNull()
        )

        | (
            (
                F.col("subscription_status")
                != "Cancelled"
            )
            & F.col("end_date").isNotNull()
        )

        | (
            (F.col("discount_percentage") > 0)
            & (
                F.col("discount_start_date").isNull()
                | F.col("discount_end_date").isNull()
            )
        )

        | (
            (F.col("discount_percentage") == 0)
            & (
                F.col("discount_start_date").isNotNull()
                | F.col("discount_end_date").isNotNull()
            )
        )

        | (
            F.col("discount_end_date").isNotNull()
            & (
                F.col("discount_end_date")
                < F.col("discount_start_date")
            )
        )
    )
    .count()
)

subscriptions_per_customer_df = (
    subscriptions_initial_df
    .groupBy("customer_id")
    .count()
)

multi_subscription_customer_count = (
    subscriptions_per_customer_df
    .filter(F.col("count") > 1)
    .count()
)

max_subscriptions_per_customer = (
    subscriptions_per_customer_df
    .agg(
        F.max("count").alias("max_count")
    )
    .first()["max_count"]
)

assert (
    actual_subscription_count
    == EXPECTED_SUBSCRIPTION_COUNT
)

assert (
    distinct_subscription_count
    == EXPECTED_SUBSCRIPTION_COUNT
)

assert customer_coverage_count == INITIAL_CUSTOMER_COUNT
assert orphan_customer_count == 0
assert subscription_before_customer_signup_count == 0
assert null_business_key_count == 0
assert invalid_price_count == 0
assert invalid_domain_value_count == 0
assert pricing_mismatch_count == 0
assert billing_amount_mismatch_count == 0
assert invalid_date_count == 0

assert (
    multi_subscription_customer_count
    == EXPECTED_MULTI_SUBSCRIPTION_CUSTOMERS
)

assert max_subscriptions_per_customer == 2

print(
    f"Generated subscriptions: "
    f"{actual_subscription_count:,}"
)

print(
    f"Distinct subscription IDs: "
    f"{distinct_subscription_count:,}"
)

print(
    f"Customers covered: "
    f"{customer_coverage_count:,}"
)

print(
    f"Orphan customer keys: "
    f"{orphan_customer_count:,}"
)

print(
    "Subscriptions before customer signup: "
    f"{subscription_before_customer_signup_count:,}"
)

print(
    f"Null business keys: "
    f"{null_business_key_count:,}"
)

print(f"Invalid prices: {invalid_price_count:,}")

print(
    f"Invalid domain values: "
    f"{invalid_domain_value_count:,}"
)

print(
    f"Pricing mismatches: "
    f"{pricing_mismatch_count:,}"
)

print(
    f"Billing amount mismatches: "
    f"{billing_amount_mismatch_count:,}"
)

print(
    f"Invalid date relationships: "
    f"{invalid_date_count:,}"
)

print(
    "Customers with multiple subscriptions: "
    f"{multi_subscription_customer_count:,}"
)

print(
    "Maximum subscriptions per customer: "
    f"{max_subscriptions_per_customer:,}"
)

display(
    subscriptions_initial_df
    .orderBy("subscription_id")
    .limit(20)
)

display(
    subscriptions_initial_df
    .groupBy(
        "plan_name",
        "subscription_status"
    )
    .count()
    .orderBy(
        "plan_name",
        "subscription_status"
    )
)

display(
    subscriptions_initial_df
    .groupBy("billing_frequency")
    .count()
    .orderBy("billing_frequency")
)

subscriptions_initial_df.printSchema()

## 5. Persist and Revalidate the Raw Subscription Snapshot

Persist the validated subscription snapshot as raw JSON files and read the files back using the explicit subscription schema.

In [0]:
# The synthetic subscription snapshot is fully regenerated on every run.
(
    subscriptions_initial_df.write
    .format("json")
    .mode("overwrite")
    .save(SUBSCRIPTIONS_INITIAL_PATH)
)

saved_subscriptions_df = (
    spark.read
    .schema(SUBSCRIPTION_SCHEMA)
    .json(SUBSCRIPTIONS_INITIAL_PATH)
)

saved_subscription_count = (
    saved_subscriptions_df.count()
)

saved_distinct_subscription_count = (
    saved_subscriptions_df
    .select("subscription_id")
    .distinct()
    .count()
)

assert (
    saved_subscription_count
    == EXPECTED_SUBSCRIPTION_COUNT
)

assert (
    saved_distinct_subscription_count
    == EXPECTED_SUBSCRIPTION_COUNT
)

print(
    f"Saved subscriptions: "
    f"{saved_subscription_count:,}"
)

print(
    f"Target path: "
    f"{SUBSCRIPTIONS_INITIAL_PATH}"
)

display(
    saved_subscriptions_df
    .orderBy("subscription_id")
    .limit(10)
)